In [1]:
from gnncloudmanufacturing.data import read_fatahi_dataset
from gnncloudmanufacturing.random_solver import random_solve
from gnncloudmanufacturing.validation import total_cost_from_graph, check_feasibility, total_cost_from_gamma
from gnncloudmanufacturing.utils import delta_from_gamma, graph_from_problem, gamma_from_target, delta_from_gamma
from gnncloudmanufacturing.gnn_solver import gnn_solve

import numpy as np
from tqdm.auto import trange, tqdm
from time import time
import pandas as pd
import torch

In [6]:
def prepare_results(params):
    results = []
    for (n_tasks, n_operations, n_cities, out_dim, n_layers) in params:
        dataset = read_fatahi_dataset(
            '../../data/fatahi.xlsx', 
            sheet_names=[
                f'{n_tasks},{n_operations},{n_cities}-1',
                f'{n_tasks},{n_operations},{n_cities}-2', 
                f'{n_tasks},{n_operations},{n_cities}-3',
            ]
        )
        problem_name = []
        total_cost = []
        comp_time = []
        for problem in dataset:
            problem_name.append(problem['name'])
            start = time()
            gamma = gnn_solve(
                problem, 
                path_to_ckpt=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
                n_operations=n_operations,
                out_dim=out_dim,
                n_layers=n_layers,
                n_iterations=30,
            )
            delta = delta_from_gamma(problem, gamma)
            check_feasibility(gamma, delta, problem)
            cost = total_cost_from_gamma(problem, gamma, delta).item()
            total_cost.append(cost)
            comp_time.append(time() - start)
        results.append(
            pd.DataFrame({'problem_name': problem_name, 'total_cost': total_cost, 'comp_time': comp_time})
        )
    return results

In [7]:
results = prepare_results([
    [ 5, 10, 10, 32, 3],
    [10, 10, 10, 32, 3],
    [ 5, 10, 20, 32, 3],
    [ 5, 20, 10, 32, 3],
    [ 5, 20, 20, 32, 3],
    [ 5,  5,  5, 16, 1],
])

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
pd.concat(results).round(2).reset_index(drop=True)

,problem_name,total_cost,comp_time
0,"5,10,10-1",5141.23,0.39
1,"5,10,10-2",7352.66,0.33
2,"5,10,10-3",7652.32,0.32
3,"10,10,10-1",15338.37,0.33
4,"10,10,10-2",14275.53,0.35
5,"10,10,10-3",14200.75,0.35
6,"5,10,20-1",4790.06,0.36
7,"5,10,20-2",5497.29,0.38
8,"5,10,20-3",5866.18,0.38
9,"5,20,10-1",14393.02,0.34
